In [3]:
import pandas as pd

In [3]:
features_list_seed2_test = []
ifn_labels_seed2_test = []
atlas_labels_seed2_test = []
min_dist_val_seed2_test = []

In [4]:
from collections import defaultdict

event_data = defaultdict(lambda: {
    'pt': [],
    'E': [],
    'rap': [],
    'phi': [],
    'flav': [],
    'ATLAS_tag': []
})

with open('ifn_status61to69_nodaughter_seed2.txt', 'r') as file:
    next(file)
    for line in file:
        columns = line.strip().split()
        event_number = int(columns[0])
        pt = float(columns[1])
        E = float(columns[2])
        rap = float(columns[3])
        phi = float(columns[4])
        flav = columns[5]
        ATLAS_tag = columns[6]
        
        event_data[event_number]['pt'].append(pt)
        event_data[event_number]['E'].append(E)
        event_data[event_number]['rap'].append(rap)
        event_data[event_number]['phi'].append(phi)
        event_data[event_number]['flav'].append(flav)
        event_data[event_number]['ATLAS_tag'].append(ATLAS_tag)

events = sorted(event_data.keys())
jet_pt_IFN = [event_data[ev]['pt'] for ev in events]
jet_E_IFN = [event_data[ev]['E'] for ev in events]
jet_eta_IFN = [event_data[ev]['rap'] for ev in events]
jet_phi_IFN = [event_data[ev]['phi'] for ev in events]
IFN_tag = [event_data[ev]['flav'] for ev in events]
ATLAS_tag = [event_data[ev]['ATLAS_tag'] for ev in events]

# Print the results (for verification)
#print("pt:", jet_pt_IFN)
'''print("E:", jet_E_IFN)
print("eta:", jet_eta_IFN)
print("phi:", jet_phi_IFN)
print("IFN_tag:", IFN_tag)
print("ATLAS_tag:", ATLAS_tag)
'''
print(jet_pt_IFN[0])

[5082.791, 4972.857, 73.943, 29.299]


In [5]:
import uproot
import numpy as np
import copy

count = 0

with uproot.open("thesis.root") as file:
    tree = file["Delphes"]

    pf_branches = [b for b in tree.keys() if b.startswith("ParticleFlowCandidate/") and not b.endswith(".fBits")]
    pf_data = {}
    for b in pf_branches:
        try:
            pf_data[b] = tree[b].array()
        except Exception as e:
            print(f"Failed to read {b}: {e}")

    jet_pt = tree["Jet/Jet.PT"].array(library="np")
    jet_eta = tree["Jet/Jet.Eta"].array(library="np")
    jet_phi = tree["Jet/Jet.Phi"].array(library="np")
    jet_particles = tree["Jet/Jet.Constituents"].array()
    
t = 0
# Number of events
n_events = len(jet_pt)
print(n_events)

10000


In [6]:
print(jet_particles[0][1])

{fName: '', fSize: 89, refs: [2451, 2431, ..., 2481]}


In [23]:
constituent_rows = []
jet_rows = []

total_cons = 0

for evt in range(n_events): 
    #print(f"\n=== Event {evt} ===")
    #print("Number of IFN jets in this event: ",len(jet_phi_IFN[evt]))

    pt_jet = jet_pt[evt][:2].tolist()
    eta_jet = jet_eta[evt][:2].tolist()
    phi_jet = jet_phi[evt][:2].tolist()
    refs_event = [[int(r) for r in jet.refs] for jet in jet_particles[evt][:2]]
    
    # if evt==0:
    #     print(refs_event)
    #print("Number of detector jets in this event: ",len(phi_jet))

    jet_phi_IFN_temp = []
    jet_eta_IFN_temp = []
    detector_jet_phi = []
    detector_jet_eta = []

    IFN_tag_detector_jets = []
    ATLAS_tag_detector_jets = []

    if len(jet_phi_IFN[evt])>=len(phi_jet):
        event_IFN_tag = IFN_tag[evt]
        event_ATLAS_tag = ATLAS_tag[evt]
        # Mapping the IFN jets to the detector jets
        jet_phi_IFN_temp = copy.deepcopy(jet_phi_IFN[evt])
        jet_eta_IFN_temp = copy.deepcopy(jet_eta_IFN[evt])
        jet_pt_IFN_temp = copy.deepcopy(jet_pt_IFN[evt])
        detector_jet_phi = copy.deepcopy(phi_jet)
        detector_jet_eta = copy.deepcopy(eta_jet)
        detector_jet_pt = copy.deepcopy(pt_jet)
        
        mapping = []
        min_dist_array = []
        
        for j in range(len(detector_jet_phi)):
            temp_array = []
            for k in range(len(jet_phi_IFN_temp)):
                d = 0
                DEL_PHI = detector_jet_phi[j]-jet_phi_IFN_temp[k]
                if DEL_PHI > np.pi:
                    DEL_PHI = DEL_PHI - 2 * np.pi
                elif DEL_PHI < -np.pi:
                    DEL_PHI = DEL_PHI + 2 * np.pi
                else:
                    DEL_PHI = DEL_PHI
                 
                d = np.sqrt((detector_jet_eta[j]-jet_eta_IFN_temp[k])**2 + DEL_PHI**2)
                temp_array.append(float(d))
            min_value = min(temp_array)  
            min_index = temp_array.index(min_value)  # Find the index of the minimum value
            mapping.append(min_index)
            min_dist_array.append(min_value)
            
        # print(mapping)
        # print("\n")
        # print(min_dist_array)
    
        for l in range(len(mapping)):
            key = mapping[l]
            min_dist_val_seed2_test.append(min_dist_array[l])
            if (min_dist_array[l]<=0.4): 
                IFN_tag_detector_jets.append(event_IFN_tag[key])
                ATLAS_tag_detector_jets.append(event_ATLAS_tag[key])
                count = count + 1
            else:
                IFN_tag_detector_jets.append(0)
                ATLAS_tag_detector_jets.append(0)

        # print(IFN_tag_detector_jets)
        # print(ATLAS_tag_detector_jets)

    else:
        print("Number of IFN jets are lower than detector jets.")
        for i in range(len(phi_jet)):
            IFN_tag_detector_jets.append(0)
            ATLAS_tag_detector_jets.append(0)

        # print(IFN_tag_detector_jets)
        # print(ATLAS_tag_detector_jets)

    # Get PFCandidates for this event
    pf_id     = pf_data["ParticleFlowCandidate/ParticleFlowCandidate.fUniqueID"][evt].to_list()
    pf_charge = pf_data["ParticleFlowCandidate/ParticleFlowCandidate.Charge"][evt].to_list()
    pf_pt     = pf_data["ParticleFlowCandidate/ParticleFlowCandidate.PT"][evt].to_list()
    pf_eta    = pf_data["ParticleFlowCandidate/ParticleFlowCandidate.Eta"][evt].to_list()
    pf_phi    = pf_data["ParticleFlowCandidate/ParticleFlowCandidate.Phi"][evt].to_list()
    pf_E      = pf_data["ParticleFlowCandidate/ParticleFlowCandidate.E"][evt].to_list()

    for j, refs in enumerate(refs_event):
        # print(f"\n  Jet {j}:")
        # print(f"    pt  = {pt_jet[j]:.2f}")
        # print(f"    eta = {eta_jet[j]:.2f}")
        # print(f"    phi = {phi_jet[j]:.2f}")
        # print(f"    Constituents: {len(refs)}")
        # print(f"    IFN_tag = {IFN_tag_detector_jets[j]}")
        # print(f"    ATLAS_tag = {ATLAS_tag_detector_jets[j]}")
        total_cons = total_cons + len(refs)

        r = {
                "event_no": evt+1,
                "jet_no": j+1,
                "IFN tag": IFN_tag_detector_jets[j],
                "ATLAS tag": ATLAS_tag_detector_jets[j],
        }
        jet_rows.append(r)  
    
        # Map jet constituents to PFCandidate data
        try:
            indices = [pf_id.index(ref) for ref in refs]
            #print(indices)
        except ValueError as e:
            print(f"    Warning: Ref not found in pf_id: {e}")
            continue

        #Calculating the jet width

        eta_cons = []
        phi_cons = []
        pt_cons = []
        charge_cons = []
        energy_cons = []
        
        for i in indices:
            eta_cons.append(pf_eta[i])
            phi_cons.append(pf_phi[i])
            pt_cons.append(pf_pt[i])
            charge_cons.append(pf_charge[i])
            energy_cons.append(pf_E[i])

            row = {
                "event_no": evt+1,
                "jet_no": j+1,
                "pt": pf_pt[i],
                "eta": pf_eta[i],
                "phi": pf_phi[i],
                "charge": pf_charge[i],
                "energy": pf_E[i]
            }
            constituent_rows.append(row)            

        # print("Constituent Informations:")
        # print("Eta: ",eta_cons, "length: ",len(eta_cons))
        # print("Phi: ",phi_cons, "length: ",len(phi_cons))
        # print("Pt: ",pt_cons, "length: ",len(pt_cons))
        # print("charge: ",charge_cons, "length: ",len(charge_cons))
        # print("energy: ",energy_cons, "length: ",len(energy_cons))

df_constituents = pd.DataFrame(constituent_rows)
df_jet = pd.DataFrame(jet_rows)
print(total_cons)

980476


In [24]:
print(df_constituents.head(5))
print(df_constituents.shape)
df_constituents.to_csv("constituents.csv", index=False)

   event_no  jet_no         pt       eta       phi  charge     energy
0         1       1   1.164495  0.351580 -2.296837       1   1.243080
1         1       1   1.721351  0.309108 -2.260369       0   1.804243
2         1       1   0.238875  2.112318 -1.977532       1   0.302687
3         1       1   1.052108  0.310202 -2.170598       0   1.103135
4         1       1  23.647419  0.357308 -2.067841       1  25.172964
(980476, 7)


In [25]:
print(df_jet.head(10))
print(df_jet.shape)

   event_no  jet_no IFN tag ATLAS tag
0         1       1       u         g
1         1       2       u         u
2         2       1    sbar      sbar
3         2       2       d         d
4         3       1       u         u
5         3       2    ubar         g
6         4       1       u         u
7         4       2       g         g
8         5       1       u         u
9         5       2       g         g
(19998, 4)


In [27]:
num_g = (df_jet["IFN tag"] == 'g').sum()
num_not_g = (df_jet["IFN tag"] != 'g').sum()

print(f"Number of g: {num_g}")
print(f"Number of not g: {num_not_g}")

Number of g: 1022
Number of not g: 18976


In [28]:
num_g = (df_jet["ATLAS tag"] == 'g').sum()
num_not_g = (df_jet["ATLAS tag"] != 'g').sum()

print(f"Number of g: {num_g}")
print(f"Number of not g: {num_not_g}")

Number of g: 5198
Number of not g: 14800


In [29]:
df_jet["IFN tag"] = df_jet["IFN tag"].apply(lambda x: 0 if x == "g" else 1)
df_jet["ATLAS tag"] = df_jet["ATLAS tag"].apply(lambda x: 0 if x == "g" else 1)

In [30]:
ifn_counts = df_jet["IFN tag"].value_counts()
atlas_counts = df_jet["ATLAS tag"].value_counts()

print("IFN tag counts:")
print(ifn_counts)

print("\nATLAS tag counts:")
print(atlas_counts)

IFN tag counts:
IFN tag
1    18976
0     1022
Name: count, dtype: int64

ATLAS tag counts:
ATLAS tag
1    14800
0     5198
Name: count, dtype: int64


In [32]:
print(df_jet.head(10))
print(df_jet.shape)
df_jet.to_csv("tags.csv", index=False)

   event_no  jet_no  IFN tag  ATLAS tag
0         1       1        1          0
1         1       2        1          1
2         2       1        1          1
3         2       2        1          1
4         3       1        1          1
5         3       2        1          0
6         4       1        1          1
7         4       2        0          0
8         5       1        1          1
9         5       2        0          0
(19998, 4)


In [4]:
df_constituents = pd.read_csv("constituents.csv")
print(df_constituents.head(100))

    event_no  jet_no          pt       eta       phi  charge      energy
0          1       1    1.164495  0.351580 -2.296837       1    1.243080
1          1       1    1.721351  0.309108 -2.260369       0    1.804243
2          1       1    0.238875  2.112318 -1.977532       1    0.302687
3          1       1    1.052108  0.310202 -2.170598       0    1.103135
4          1       1   23.647419  0.357308 -2.067841       1   25.172964
..       ...     ...         ...       ...       ...     ...         ...
95         1       2   78.056458 -0.338808  1.131371       0   82.579559
96         1       2   10.668327 -0.338444  1.094758      -1   11.285836
97         1       2  189.579239 -0.334410  1.100577       1  200.278748
98         1       2   13.613062 -0.332747  1.109482       1   14.374199
99         1       2   85.287544 -0.349119  1.100915      -1   90.539467

[100 rows x 7 columns]


In [6]:
df_jet = pd.read_csv("tags.csv")
print(df_jet.head(10))

   event_no  jet_no  IFN tag  ATLAS tag
0         1       1        1          0
1         1       2        1          1
2         2       1        1          1
3         2       2        1          1
4         3       1        1          1
5         3       2        1          0
6         4       1        1          1
7         4       2        0          0
8         5       1        1          1
9         5       2        0          0
